1. **Cargar datos procesados** desde `artifacts/datos_procesados.pkl`.
2. **Entrenar modelos base** (LogisticRegression, RandomForest, XGBoost).
3. **Evaluar con métricas** (accuracy, precision, recall, f1, roc\_auc).
4. **Guardar modelos** y **métricas**.
5. **Visualizar resultados**:

   * Tabla comparativa.
   * Gráficas ROC.
   * Importancia de variables (RandomForest / XGBoost).

In [1]:
# --- Configuración inicial ---
import pandas as pd
import sys
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)


from sklearn.feature_selection import SelectFromModel, RFE
from imblearn.over_sampling import SMOTE

import shap

# --- Rutas ---
ARTIFACTS_DIR = "artifacts"
MODELOS_DIR = os.path.join(ARTIFACTS_DIR, "modelos")
RESULTADOS_PATH = os.path.join(ARTIFACTS_DIR, "resultados_modelos.csv")
DATOS_PROCESADOS_PATH = os.path.join("data", "processed", "datos_procesados.pkl")

os.makedirs(MODELOS_DIR, exist_ok=True)

# 🔹 Forzar a que siempre arranque desde la raíz del repo
os.chdir("..")  # sube a la raíz



In [ ]:
os.getcwd()

### 🔹 Cargar datos procesados


In [ ]:
with open(DATOS_PROCESADOS_PATH, "rb") as f:
    X_train, X_test, y_train, y_test = pickle.load(f)

print("Shapes:")
print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)

### 🔹 Definir modelos

In [4]:
modelos = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric="logloss", random_state=42)
}

### 🔹 Entrenamiento y evaluación

In [ ]:


# Codificar las etiquetas
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print("Clases originales:", le.classes_)  # ['Canceló' 'Retenido']

resultados = []
fprs, tprs = {}, {}

for nombre, modelo in modelos.items():
    print(f"Entrenando modelo: {nombre}...")
    modelo.fit(X_train, y_train_enc)  # usar codificado

    y_pred_enc = modelo.predict(X_test)
    y_pred = le.inverse_transform(y_pred_enc)  # volver a texto para métricas

    y_prob = modelo.predict_proba(X_test)[:, 1] if hasattr(modelo, "predict_proba") else None

    metrics = {
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, pos_label="Canceló"),
        "recall": recall_score(y_test, y_pred, pos_label="Canceló"),
        "f1": f1_score(y_test, y_pred, pos_label="Canceló"),
        "roc_auc": roc_auc_score((y_test == "Canceló").astype(int), y_prob) if y_prob is not None else None
    }
    resultados.append(metrics)

    # Guardar modelo entrenado
    modelo_path = os.path.join(MODELOS_DIR, f"{nombre}.pkl")
    with open(modelo_path, "wb") as f:
        pickle.dump(modelo, f)

    # Guardar curvas ROC
    if y_prob is not None:
        fpr, tpr, _ = roc_curve((y_test == "Canceló").astype(int), y_prob)
        fprs[nombre], tprs[nombre] = fpr, tpr

df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv(RESULTADOS_PATH, index=False)
df_resultados


### 🔹 Curvas ROC

In [ ]:
plt.figure(figsize=(8,6))
for nombre in fprs:
    plt.plot(fprs[nombre], tprs[nombre], label=nombre)
plt.plot([0,1], [0,1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curvas ROC")
plt.legend()
plt.show()

### 🔹 Importancia de variables (RandomForest / XGBoost)

In [ ]:
## === Cargar nombres de variables (feature_names) ===
import pickle
import os

FEATURE_NAMES_PATH = os.path.join("data", "processed", "feature_names.pkl")

with open(FEATURE_NAMES_PATH, "rb") as f:
    feature_names = pickle.load(f)

print(f"Se cargaron {len(feature_names)} variables desde feature_names.pkl")
print("Ejemplo de variables:", feature_names[:10])


In [ ]:
# RandomForest
if "RandomForest" in modelos:
    rf = modelos["RandomForest"]
    importancias = rf.feature_importances_
    idx = np.argsort(importancias)[::-1][:20]  # top 20
    
    plt.barh(range(len(idx)), importancias[idx])
    plt.yticks(range(len(idx)), [feature_names[i] for i in idx])
    plt.gca().invert_yaxis()
    plt.title("Importancia de variables - RandomForest (Top 20)")
    plt.show()

# XGBoost
if "XGBoost" in modelos:
    xgb = modelos["XGBoost"]
    importances = xgb.feature_importances_
    idx = np.argsort(importances)[::-1][:20]  # top 20
    
    plt.barh(range(len(idx)), importances[idx])
    plt.yticks(range(len(idx)), [feature_names[i] for i in idx])
    plt.gca().invert_yaxis()
    plt.title("Importancia de variables - XGBoost (Top 20)")
    plt.show()

# Regresión Logística (coeficientes absolutos)
if "LogisticRegression" in modelos:
    log_reg = modelos["LogisticRegression"]
    coefs = np.abs(log_reg.coef_[0])  # tomamos valor absoluto
    idx = np.argsort(coefs)[::-1][:20]
    
    plt.barh(range(len(idx)), coefs[idx])
    plt.yticks(range(len(idx)), [feature_names[i] for i in idx])
    plt.gca().invert_yaxis()
    plt.title("Importancia de variables - Regresión Logística (Top 20)")
    plt.show()



# 📓 Análisis de Importancia de Variables y Balanceo de Modelos

## 🔹 Paso 1. Diagnóstico Profundo

In [ ]:

# === Revisar top variables de los modelos entrenados (adaptado a feature_names.pkl) ===

# Cargar feature_names
FEATURE_NAMES_PATH = os.path.join("data", "processed", "feature_names.pkl")

with open(FEATURE_NAMES_PATH, "rb") as f:
    feature_names = pickle.load(f)

# Revisar top variables
for nombre, modelo in modelos.items():
    if hasattr(modelo, "feature_importances_"):
        importancias = modelo.feature_importances_
        idx = np.argsort(importancias)[::-1][:10]
        print(f"\nModelo: {nombre}")
        for i in idx:
            if i < len(feature_names):  # seguridad por si hay desajuste
                print(f" - {feature_names[i]}: {importancias[i]:.4f}")
            else:
                print(f" - [Index {i} fuera de rango en feature_names]: {importancias[i]:.4f}")
    elif hasattr(modelo, "coef_"):
        coefs = np.abs(modelo.coef_[0])
        idx = np.argsort(coefs)[::-1][:10]
        print(f"\nModelo: {nombre} (coeficientes)")
        for i in idx:
            if i < len(feature_names):
                print(f" - {feature_names[i]}: {coefs[i]:.4f}")
            else:
                print(f" - [Index {i} fuera de rango en feature_names]: {coefs[i]:.4f}")



In [ ]:

# === 1.2 SHAP optimizado (fix binaria y multiclase) ===
import shap
import numpy as np
import pickle
import os

# Cargar feature_names
FEATURE_NAMES_PATH = os.path.join("data", "processed", "feature_names.pkl")
with open(FEATURE_NAMES_PATH, "rb") as f:
    feature_names = pickle.load(f)

# Convertir a denso si es sparse
if hasattr(X_test, "toarray"):
    X_test_dense = X_test.toarray().astype(np.float32)
else:
    X_test_dense = np.array(X_test, dtype=np.float32)

# Submuestreo para que no explote
n_muestra = min(200, X_test_dense.shape[0])  # puedes ajustar 200 → 100 si sigue pesado
idx_sample = np.random.choice(X_test_dense.shape[0], n_muestra, replace=False)
X_sample = X_test_dense[idx_sample, :]

# Verificamos longitud de nombres de features
if len(feature_names) != X_sample.shape[1]:
    feature_names = [f"feature_{i}" for i in range(X_sample.shape[1])]
    print("⚠️ Advertencia: feature_names reemplazados por índices automáticos.")

# Calcular SHAP
for nombre, modelo in modelos.items():
    if nombre in ["RandomForest", "XGBoost"]:
        print(f"\nExplicabilidad con SHAP para {nombre} (sample={n_muestra})")

        # Usamos API moderna
        explainer = shap.Explainer(modelo, X_sample)
        shap_values = explainer(X_sample)

        # Si es clasificación multiclase → tomar clase positiva (1)
        if len(shap_values.values.shape) == 3:
            shap_values = shap_values[:, :, 1]

        # Resumen general
        shap.summary_plot(
            shap_values,
            X_sample,
            feature_names=feature_names,
            max_display=15,
            show=True
        )

        # Explicación individual (primer registro)
        shap.plots.waterfall(shap_values[0], max_display=10)






In [ ]:
# === 1.3 Explicabilidad con SHAP para modelos lineales (ejemplo: Regresión Logística) ===
if "LogisticRegression" in modelos:
    print("\nExplicabilidad con SHAP para Regresión Logística")

    lr = modelos["LogisticRegression"]

    # Convertir X_train / X_test a denso si son sparse
    if hasattr(X_train, "toarray"):
        X_train_dense = X_train.toarray().astype(np.float32)
        X_test_dense = X_test.toarray().astype(np.float32)
    else:
        X_train_dense = np.array(X_train, dtype=np.float32)
        X_test_dense = np.array(X_test, dtype=np.float32)

    # Verificamos longitud de nombres de features
    if len(feature_names) != X_test_dense.shape[1]:
        feature_names = [f"feature_{i}" for i in range(X_test_dense.shape[1])]
        print("⚠️ Advertencia: feature_names reemplazados por índices automáticos.")

    # Usar Explainer moderno
    explainer = shap.Explainer(lr, X_train_dense)
    shap_values = explainer(X_test_dense)

    # Si es clasificación binaria → tomar clase positiva (1)
    if len(shap_values.values.shape) == 3:
        shap_values = shap_values[:, :, 1]

    # Resumen general
    shap.summary_plot(
        shap_values,
        X_test_dense,
        feature_names=feature_names,
        max_display=15,
        show=True
    )

    # Explicación individual (primer registro)
    shap.plots.waterfall(shap_values[0], max_display=10)



## 🔹 Paso 2. Acción Correctiva Principal

In [12]:
dff = pd.read_csv("data/processed/train_ready.csv")
dff.head()

,customer_tenure,cargo_mensual,id_cliente_0002-ORFBO,id_cliente_0004-TLHLJ,id_cliente_0013-EXCHZ,id_cliente_0013-MHZWF,id_cliente_0013-SMEOE,id_cliente_0014-BMAQU,id_cliente_0015-UOCOJ,id_cliente_0016-QLJIS,...,cargo_total_995.35,cargo_total_996.45,cargo_total_996.85,cargo_total_996.95,cargo_total_997.65,cargo_total_997.75,cargo_total_999.45,cargo_total_999.8,cargo_total_999.9,cancelacion
0,-1.071436,-0.493018,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Retenido
1,0.231010,1.334258,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Canceló
2,1.289247,0.373234,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Retenido
3,0.882232,1.299341,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Canceló
4,-1.112137,-0.301810,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Retenido


In [13]:

import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

# === 2.1 Reentrenar con balanceo de clases ===

# Paths
TRAIN_PATH = os.path.join("data", "processed", "train_ready.csv")
TEST_PATH = os.path.join("data", "processed", "test_ready.csv")

# Cargar datasets listos
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

# Definir columna objetivo
TARGET = "cancelacion"

# Convertir target a binario: 1 = Canceló, 0 = Retenido
df_train[TARGET] = df_train[TARGET].map({"Canceló": 1, "Retenido": 0})
df_test[TARGET] = df_test[TARGET].map({"Canceló": 1, "Retenido": 0})

# Separar features y target
X_train = df_train.drop(columns=[TARGET])
y_train = df_train[TARGET]
X_test = df_test.drop(columns=[TARGET])
y_test = df_test[TARGET]

# RandomForest balanceado
rf_bal = RandomForestClassifier(class_weight="balanced", random_state=42)
rf_bal.fit(X_train, y_train)

# XGBoost balanceado
scale_pos = y_train.value_counts()[0] / y_train.value_counts()[1]
xgb_bal = XGBClassifier(
    scale_pos_weight=scale_pos,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
xgb_bal.fit(X_train, y_train)

# LogisticRegression balanceada
log_bal = LogisticRegression(
    class_weight="balanced",
    solver="liblinear",
    penalty="l2",
    random_state=42
)
log_bal.fit(X_train, y_train)



/home/david/Predicci-n-de-Cancelaci-n-Churn-/venv_churn/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [23:21:31] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [14]:

from sklearn.metrics import classification_report, roc_auc_score

# === 2.2 Evaluación inicial ===
modelos_balanceados = {
    "RandomForest": rf_bal,
    "XGBoost": xgb_bal,
    "LogisticRegression": log_bal
}

for nombre, modelo in modelos_balanceados.items():
    y_pred = modelo.predict(X_test)
    print(f"\n--- {nombre} ---")
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1]))





--- RandomForest ---
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409

ROC-AUC: 0.8334663773282699

--- XGBoost ---
              precision    recall  f1-score   support

           0       0.88      0.79      0.83      1035
           1       0.55      0.70      0.62       374

    accuracy                           0.77      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.79      0.77      0.78      1409

ROC-AUC: 0.8326526130873957

--- LogisticRegression ---
              precision    recall  f1-score   support

           0       0.90      0.79      0.84      1035
           1       0.57      0.75      0.65       374

    accuracy                           0.78      1409
   ma

In [ ]:

from imblearn.over_sampling import SMOTE

# === 2.3 Probar SMOTE si es necesario ===
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

rf_smote = RandomForestClassifier(random_state=42)
rf_smote.fit(X_res, y_res)

print("\nROC-AUC RF+SMOTE:", roc_auc_score(y_test, rf_smote.predict_proba(X_test)[:, 1]))


## 🔹 Paso 3. Refinamiento y Simplificación

In [ ]:

from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# === 3.1 Selección de variables con RandomForest ===
sel = SelectFromModel(
    RandomForestClassifier(random_state=42), 
    threshold="median"
)
sel.fit(X_train, y_train)

# Transformar datasets
X_train_sel = sel.transform(X_train)
X_test_sel = sel.transform(X_test)

# Mostrar variables seleccionadas
selected_features = X_train.columns[sel.get_support()]
print("Variables seleccionadas:", list(selected_features))
print(f"Total seleccionadas: {len(selected_features)} de {X_train.shape[1]}")


In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# === 3.2 Regularización en Regresión Logística ===

# Regresión Logística con L1 (Lasso)
log_l1 = LogisticRegression(
    penalty="l1", 
    solver="liblinear", 
    class_weight="balanced", 
    random_state=42,
    max_iter=1000
)
log_l1.fit(X_train_sel, y_train)

# Regresión Logística con L2 (Ridge)
log_l2 = LogisticRegression(
    penalty="l2", 
    solver="liblinear", 
    class_weight="balanced", 
    random_state=42,
    max_iter=1000
)
log_l2.fit(X_train_sel, y_train)

# Evaluación con ROC-AUC
print("ROC-AUC L1:", roc_auc_score(y_test, log_l1.predict_proba(X_test_sel)[:, 1]))
print("ROC-AUC L2:", roc_auc_score(y_test, log_l2.predict_proba(X_test_sel)[:, 1]))


## 🔹 Paso 4. Interpretabilidad Final con SHAP

In [ ]:

import shap

# === 4. Interpretabilidad Final con SHAP ===

# Aseguramos que X_test tenga formato numpy
X_test_array = X_test.values if hasattr(X_test, "values") else X_test

# ---------------------------
# 4.1 RandomForest balanceado
# ---------------------------
print("\nSHAP para RandomForest balanceado")
explainer_rf = shap.TreeExplainer(rf_bal)
shap_values_rf = explainer_rf.shap_values(X_test_array)

# Gráfico resumen (barras)
shap.summary_plot(
    shap_values_rf, 
    X_test_array, 
    feature_names=feature_names, 
    plot_type="bar", 
    show=True
)

# Gráfico detallado
shap.summary_plot(
    shap_values_rf, 
    X_test_array, 
    feature_names=feature_names, 
    show=True
)


# ---------------------------
# 4.2 XGBoost balanceado
# ---------------------------
print("\nSHAP para XGBoost balanceado")
explainer_xgb = shap.TreeExplainer(xgb_bal)
shap_values_xgb = explainer_xgb.shap_values(X_test_array)

# Gráfico resumen (barras)
shap.summary_plot(
    shap_values_xgb, 
    X_test_array, 
    feature_names=feature_names, 
    plot_type="bar", 
    show=True
)

# Gráfico detallado
shap.summary_plot(
    shap_values_xgb, 
    X_test_array, 
    feature_names=feature_names, 
    show=True
)


# ---------------------------
# 4.3 LogisticRegression balanceado
# ---------------------------
print("\nSHAP para LogisticRegression balanceado")
explainer_lr = shap.LinearExplainer(log_bal, X_train, feature_names=feature_names)
shap_values_lr = explainer_lr(X_test_array)

# Gráfico resumen (barras)
shap.summary_plot(
    shap_values_lr, 
    X_test_array, 
    feature_names=feature_names, 
    plot_type="bar", 
    show=True
)

# Gráfico detallado
shap.summary_plot(
    shap_values_lr, 
    X_test_array, 
    feature_names=feature_names, 
    show=True
)
